In [1]:
from pathlib import Path

import numpy as np

from zmax_datasets.datasets.usleep import USleepDataset

DATASETS_DIR = Path("/project/4180000.48/sleep_staging/sleep_datasets/processed")
DATASET_NAME = "shhs"

dataset = USleepDataset(data_dir=DATASETS_DIR / DATASET_NAME)
print(dataset.n_recordings)
print(dataset.recording_ids)

3506
['shhs1-200001', 'shhs1-200002', 'shhs1-200003', 'shhs1-200004', 'shhs1-200005', 'shhs1-200006', 'shhs1-200007', 'shhs1-200008', 'shhs1-200033', 'shhs1-200043', 'shhs1-200017', 'shhs1-200038', 'shhs1-200031', 'shhs1-200034', 'shhs1-200028', 'shhs1-200032', 'shhs1-200037', 'shhs1-200023', 'shhs1-200025', 'shhs1-200036', 'shhs1-200018', 'shhs1-200040', 'shhs1-200047', 'shhs1-200011', 'shhs1-200024', 'shhs1-200045', 'shhs1-200027', 'shhs1-200039', 'shhs1-200012', 'shhs1-200015', 'shhs1-200041', 'shhs1-200026', 'shhs1-200029', 'shhs1-200022', 'shhs1-200046', 'shhs1-200021', 'shhs1-200044', 'shhs1-200009', 'shhs1-200013', 'shhs1-200035', 'shhs1-200042', 'shhs1-200019', 'shhs1-200016', 'shhs1-200048', 'shhs1-200010', 'shhs1-200030', 'shhs1-200014', 'shhs1-200049', 'shhs1-200020', 'shhs1-200086', 'shhs1-200059', 'shhs1-200061', 'shhs1-200050', 'shhs1-200062', 'shhs1-200051', 'shhs1-200054', 'shhs1-200052', 'shhs1-200085', 'shhs1-200056', 'shhs1-200064', 'shhs1-200068', 'shhs1-200075', 's

In [2]:
sample_recording = dataset.get_recording("shhs1-200082")
print(sample_recording.data_types)

{'ECG_artifactual_peaks': DataType(channel='ECG_artifactual_peaks', sampling_rate=128.0), 'ECG_filtered': DataType(channel='ECG_filtered', sampling_rate=128.0), 'ECG_hr': DataType(channel='ECG_hr', sampling_rate=128.0), 'ECG_ibi': DataType(channel='ECG_ibi', sampling_rate=128.0), 'ECG_peaks': DataType(channel='ECG_peaks', sampling_rate=128.0)}


In [5]:
from zmax_datasets.exports.utils import SleepAnnotations

SIGNAL = "ECG"
WITH_QUALITY = False
WITH_ARTIFACTUAL_PEAKS = True
HAS_ANNOTATIONS = True
HAS_ARTIFACT_LABELS = False
annotations = None

# Load all PPG signals
filtered = sample_recording.read_data_type(f"{SIGNAL}_filtered")
peaks = sample_recording.read_data_type(f"{SIGNAL}_peaks")
rate = sample_recording.read_data_type(f"{SIGNAL}_hr")
ibi = sample_recording.read_data_type(f"{SIGNAL}_ibi")

if WITH_QUALITY:
    quality = sample_recording.read_data_type(f"{SIGNAL}_quality")

if WITH_ARTIFACTUAL_PEAKS:
    artifactual_peaks = sample_recording.read_data_type(f"{SIGNAL}_artifactual_peaks")

# Load sleep stage annotations
try:
    annotations = sample_recording.read_annotations(SleepAnnotations.SLEEP_STAGE)
    HAS_ANNOTATIONS = True
    print(
        f"Loaded sleep stage annotations with sample rate: {annotations.sample_rate:.4f} Hz"
    )
except Exception as e:
    print(f"No annotations available: {e}")

if HAS_ARTIFACT_LABELS:
    artifact_labels = sample_recording.read_data_type(f"{SIGNAL}_artifact")
    print(
        f"Loaded artifact labels with sample rate: {artifact_labels.sample_rate:.4f} Hz"
    )

# Create time axis in seconds for the signals
duration_seconds = len(rate.array.squeeze()) / rate.sample_rate
time = np.linspace(0, duration_seconds, len(rate.array.squeeze()))
print(
    f"Signal duration: {duration_seconds:.1f} seconds ({duration_seconds / 60:.1f} minutes)"
)

2025-11-01 16:44:25.674 | INFO     | zmax_datasets.datasets.base:read_data_type:36 - Reading data type: ECG_filtered
2025-11-01 16:44:26.152 | INFO     | zmax_datasets.datasets.base:read_data_type:36 - Reading data type: ECG_peaks
2025-11-01 16:44:26.405 | INFO     | zmax_datasets.datasets.base:read_data_type:36 - Reading data type: ECG_hr
2025-11-01 16:44:27.179 | INFO     | zmax_datasets.datasets.base:read_data_type:36 - Reading data type: ECG_ibi
2025-11-01 16:44:28.091 | INFO     | zmax_datasets.datasets.base:read_data_type:36 - Reading data type: ECG_artifactual_peaks


Loaded sleep stage annotations with sample rate: 0.0333 Hz
Signal duration: 32070.0 seconds (534.5 minutes)


In [6]:
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Create widgets for controlling the window
start_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=duration_seconds - 60,  # Leave room for the window
    step=10,
    description="Start Time (s):",
    style={"description_width": "initial"},
    layout={"width": "500px"},
)

window_size = widgets.Dropdown(
    options=[
        ("10 seconds", 10),
        ("30 seconds", 30),
        ("1 minute", 60),
        ("5 minutes", 300),
        ("10 minutes", 600),
        ("30 minutes", 1800),
        ("1 hour", 3600),
    ],
    value=60,
    description="Window Size:",
    style={"description_width": "initial"},
)

# Configure subplot layout based on WITH_QUALITY and HAS_ANNOTATIONS
if WITH_QUALITY and HAS_ANNOTATIONS:
    n_rows = 5
    row_heights = [0.35, 0.15, 0.15, 0.15, 0.2]
    subplot_titles = (
        f"{SIGNAL} Signal and Peaks",
        "Signal Quality",
        "Sleep Stage",
        "Heart Rate",
        "Inter-Beat Intervals",
    )
    annotation_row = 3
    hr_row = 4
    ibi_row = 5
elif WITH_QUALITY:
    n_rows = 4
    row_heights = [0.4, 0.2, 0.2, 0.2]
    subplot_titles = (
        f"{SIGNAL} Signal and Peaks",
        "Signal Quality",
        "Heart Rate",
        "Inter-Beat Intervals",
    )
    hr_row = 3
    ibi_row = 4
elif HAS_ANNOTATIONS:
    n_rows = 4
    row_heights = [0.4, 0.15, 0.225, 0.225]
    subplot_titles = (
        f"{SIGNAL} Signal and Peaks",
        "Sleep Stage",
        "Heart Rate",
        "Inter-Beat Intervals",
    )
    annotation_row = 2
    hr_row = 3
    ibi_row = 4
else:
    n_rows = 3
    row_heights = [0.5, 0.25, 0.25]
    subplot_titles = (
        f"{SIGNAL} Signal and Peaks",
        "Heart Rate",
        "Inter-Beat Intervals",
    )
    hr_row = 2
    ibi_row = 3

# Create initial figure
fig = go.FigureWidget(
    make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=row_heights,
        subplot_titles=subplot_titles,
    )
)

# Initialize with empty traces
fig.add_trace(
    go.Scatter(name=f"{SIGNAL} Filtered", line=dict(color="blue")), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        name="Peaks",
        mode="markers",
        marker=dict(color="green", size=8, symbol="circle"),
    ),
    row=1,
    col=1,
)
if WITH_ARTIFACTUAL_PEAKS:
    fig.add_trace(
        go.Scatter(
            name="Artifactual Peaks",
            mode="markers",
            marker=dict(color="red", size=8, symbol="circle"),
        ),
        row=1,
        col=1,
    )

if HAS_ARTIFACT_LABELS:
    fig.add_trace(
        go.Scatter(
            name="Artifact Segments",
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor="rgba(255, 255, 0, 0.3)",
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

if WITH_QUALITY:
    fig.add_trace(
        go.Scatter(name="Signal Quality", line=dict(color="purple"), fill="tozeroy"),
        row=2,
        col=1,
    )

if HAS_ANNOTATIONS:
    fig.add_trace(
        go.Scatter(
            name="Sleep Stage", line=dict(color="blue", shape="hv"), mode="lines"
        ),
        row=annotation_row,
        col=1,
    )

fig.add_trace(
    go.Scatter(name="Heart Rate", line=dict(color="orange")), row=hr_row, col=1
)
fig.add_trace(
    go.Scatter(name="Mean HR", line=dict(color="red", dash="dash")), row=hr_row, col=1
)
# Add IBI trace and threshold lines
fig.add_trace(go.Scatter(name="IBI", line=dict(color="green")), row=ibi_row, col=1)
fig.add_trace(
    go.Scatter(name="Min Threshold", line=dict(color="red", dash="dash")),
    row=ibi_row,
    col=1,
)
fig.add_trace(
    go.Scatter(name="Max Threshold", line=dict(color="red", dash="dash")),
    row=ibi_row,
    col=1,
)

# Update layout
# Adjust height based on number of rows
if WITH_QUALITY and HAS_ANNOTATIONS:
    layout_height = 1200
elif WITH_QUALITY or HAS_ANNOTATIONS:
    layout_height = 1000
else:
    layout_height = 900

fig.update_layout(
    height=layout_height,
    title=f"{SIGNAL} Signal Analysis - Recording {str(sample_recording)}",
    showlegend=True,
    template="plotly_white",
)

# Update axes labels
fig.update_yaxes(title_text="Amplitude", row=1, col=1)
if WITH_QUALITY:
    fig.update_yaxes(title_text="Quality Score", range=[0, 1], row=2, col=1)
if HAS_ANNOTATIONS:
    # Set y-axis as categorical for sleep stages
    fig.update_yaxes(
        title_text="Sleep Stage",
        row=annotation_row,
        col=1,
        tickmode="array",
        tickvals=[0, 1, 2, 3, 4],
        ticktext=["W", "N1", "N2", "N3", "R"],
        range=[-0.5, 4.5],  # Fixed range to always show all stages
    )
fig.update_yaxes(title_text="Heart Rate (BPM)", range=[0, 200], row=hr_row, col=1)
fig.update_yaxes(
    title_text="IBI (ms)", range=[0, 2500], row=ibi_row, col=1
)  # Set range slightly above max threshold
fig.update_xaxes(title_text="Time (seconds)", row=n_rows, col=1)


def update_plot(start_time, window_duration):
    # Calculate indices for the window using peaks/filtered sample rate (they should match)
    signal_sr = filtered.sample_rate
    start_idx = int(start_time * signal_sr)
    end_idx = int((start_time + window_duration) * signal_sr)

    # Create time array for the window
    time_window = np.linspace(
        start_time, start_time + window_duration, end_idx - start_idx
    )

    # Get signal segments
    filtered_signal = filtered.array.squeeze()[start_idx:end_idx]
    peaks_signal = peaks.array.squeeze()[start_idx:end_idx]
    if WITH_ARTIFACTUAL_PEAKS:
        # Artifacts should have the same sampling rate as peaks/filtered
        artifactual_peaks_signal = artifactual_peaks.array.squeeze()[start_idx:end_idx]
    if WITH_QUALITY:
        quality_signal = quality.array.squeeze()[start_idx:end_idx]
    if HAS_ANNOTATIONS:
        # Get annotation indices for the time window
        annotation_start_idx = int(start_time * annotations.sample_rate)
        annotation_end_idx = int(
            (start_time + window_duration) * annotations.sample_rate
        )
        annotation_signal = annotations.array.squeeze()[
            annotation_start_idx:annotation_end_idx
        ]
        annotation_time = np.linspace(
            start_time, start_time + window_duration, len(annotation_signal)
        )

    if HAS_ARTIFACT_LABELS:
        # Get artifact label indices for the time window
        artifact_start_idx = int(start_time * artifact_labels.sample_rate)
        artifact_end_idx = int(
            (start_time + window_duration) * artifact_labels.sample_rate
        )
        artifact_signal = artifact_labels.array.squeeze()[
            artifact_start_idx:artifact_end_idx
        ]
        # Compute time from indices using the sample rate (not linspace which assumes continuous data)
        artifact_time = (
            np.arange(artifact_start_idx, artifact_start_idx + len(artifact_signal))
            / artifact_labels.sample_rate
        )

    # Rate and IBI might have different sample rates
    rate_start_idx = int(start_time * rate.sample_rate)
    rate_end_idx = int((start_time + window_duration) * rate.sample_rate)
    rate_signal = rate.array.squeeze()[rate_start_idx:rate_end_idx]
    rate_time_window = np.linspace(
        start_time, start_time + window_duration, len(rate_signal)
    )

    ibi_signal = ibi.array.squeeze()[
        start_idx:end_idx
    ]  # IBI typically matches filtered signal rate

    # Determine trace indices based on WITH_ARTIFACTS, WITH_QUALITY and HAS_ANNOTATIONS
    trace_idx = 2  # Starting after signal and peaks (indices 0 and 1)
    if WITH_ARTIFACTUAL_PEAKS:
        artifactual_peaks_trace_idx = trace_idx
        trace_idx += 1
    if HAS_ARTIFACT_LABELS:
        artifact_labels_trace_idx = trace_idx
        trace_idx += 1
    if WITH_QUALITY:
        quality_trace_idx = trace_idx
        trace_idx += 1
    if HAS_ANNOTATIONS:
        annotation_trace_idx = trace_idx
        trace_idx += 1
    hr_trace_idx = trace_idx
    mean_hr_trace_idx = trace_idx + 1
    ibi_trace_idx = trace_idx + 2
    min_threshold_trace_idx = trace_idx + 3
    max_threshold_trace_idx = trace_idx + 4

    # Update traces with new data
    with fig.batch_update():
        # Update PPG signal
        fig.data[0].x = time_window
        fig.data[0].y = filtered_signal

        # Update peaks - separate normal peaks from artifact peaks
        peaks_idx = np.where(peaks_signal == 1)[0]
        if WITH_ARTIFACTUAL_PEAKS:
            # Check that artifactual_peaks_signal has the same length
            if len(artifactual_peaks_signal) != len(peaks_signal):
                print(
                    f"Warning: artifactual_peaks length ({len(artifactual_peaks_signal)}) != peaks length ({len(peaks_signal)})"
                )

            # Separate peaks into normal and artifact peaks
            artifact_peaks_idx = []
            normal_peaks_idx = []
            for idx in peaks_idx:
                # Make sure idx is within bounds
                if (
                    idx < len(artifactual_peaks_signal)
                    and artifactual_peaks_signal[idx] == 1
                ):
                    artifact_peaks_idx.append(idx)
                else:
                    normal_peaks_idx.append(idx)

            # Update normal peaks (trace index 1)
            if len(normal_peaks_idx) > 0:
                normal_peaks_idx = np.array(normal_peaks_idx)
                fig.data[1].x = time_window[normal_peaks_idx]
                fig.data[1].y = filtered_signal[normal_peaks_idx]
            else:
                fig.data[1].x = []
                fig.data[1].y = []

            # Update artifact peaks
            if len(artifact_peaks_idx) > 0:
                artifact_peaks_idx = np.array(artifact_peaks_idx)
                fig.data[artifactual_peaks_trace_idx].x = time_window[
                    artifact_peaks_idx
                ]
                fig.data[artifactual_peaks_trace_idx].y = filtered_signal[
                    artifact_peaks_idx
                ]
                # Debug: print number of artifact peaks found
                print(
                    f"Found {len(artifact_peaks_idx)} artifactual peaks out of {len(peaks_idx)} total peaks"
                )
            else:
                fig.data[artifactual_peaks_trace_idx].x = []
                fig.data[artifactual_peaks_trace_idx].y = []
                if len(peaks_idx) > 0:
                    print(
                        f"No artifactual peaks found (checked {len(peaks_idx)} peaks, artifactual peaks signal has {np.sum(artifactual_peaks_signal)} ones)"
                    )
        else:
            # No artifacts - show all peaks in green
            if len(peaks_idx) > 0:
                fig.data[1].x = time_window[peaks_idx]
                fig.data[1].y = filtered_signal[peaks_idx]
            else:
                fig.data[1].x = []
                fig.data[1].y = []

        # Update artifact labels overlay (only if HAS_ARTIFACT_LABELS is True)
        if HAS_ARTIFACT_LABELS:
            # Create overlay regions for artifact segments
            # artifact_signal is binary (1 = artifact, 0 = clean)
            # We'll create a filled area that spans the y-range where artifacts exist
            y_min, y_max = np.min(filtered_signal), np.max(filtered_signal)

            # Build the overlay as a shaded region
            # Use a flat line at y_max that fills down to zero
            overlay_x = []
            overlay_y = []
            segment_duration = 1.0 / artifact_labels.sample_rate  # e.g., 30 seconds

            for i in range(len(artifact_signal)):
                if artifact_signal[i] == 1:  # Artifact segment
                    segment_start_time = artifact_time[i]
                    segment_end_time = segment_start_time + segment_duration
                    # Clip to window boundaries
                    segment_start_time = max(segment_start_time, start_time)
                    segment_end_time = min(
                        segment_end_time, start_time + window_duration
                    )
                    # Add rectangle vertices: bottom-left, bottom-right, top-right, top-left, bottom-left (to close)
                    overlay_x.extend(
                        [
                            segment_start_time,
                            segment_end_time,
                            segment_end_time,
                            segment_start_time,
                            segment_start_time,
                        ]
                    )
                    overlay_y.extend([y_min, y_min, y_max, y_max, y_min])
                    # Only add NaN if next segment is not consecutive (to prevent connecting separate regions)
                    if i < len(artifact_signal) - 1 and artifact_signal[i + 1] != 1:
                        overlay_x.append(np.nan)
                        overlay_y.append(np.nan)

            fig.data[artifact_labels_trace_idx].x = overlay_x if overlay_x else []
            fig.data[artifact_labels_trace_idx].y = overlay_y if overlay_y else []

        # Update signal quality (only if WITH_QUALITY is True)
        if WITH_QUALITY:
            fig.data[quality_trace_idx].x = time_window
            fig.data[quality_trace_idx].y = quality_signal
            mean_quality = np.nanmean(quality_signal)
            fig.data[quality_trace_idx].name = f"Signal Quality ({mean_quality:.2f})"

        # Update sleep stage annotations (only if HAS_ANNOTATIONS is True)
        if HAS_ANNOTATIONS:
            # Map sleep stage labels to numeric values for visualization
            stage_mapping = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4, "UNKNOWN": -1}
            numeric_annotations = np.array(
                [stage_mapping.get(stage, -1) for stage in annotation_signal]
            )

            # For hypnogram, we need to duplicate points to create step effect
            if len(numeric_annotations) > 0:
                # Create step coordinates for hypnogram
                step_x = []
                step_y = []

                for i in range(len(numeric_annotations)):
                    # Add start point
                    step_x.append(annotation_time[i])
                    step_y.append(numeric_annotations[i])
                    # Add end point (next epoch start)
                    if i < len(numeric_annotations) - 1:
                        step_x.append(annotation_time[i + 1])
                        step_y.append(numeric_annotations[i])

                fig.data[annotation_trace_idx].x = step_x
                fig.data[annotation_trace_idx].y = step_y
            else:
                fig.data[annotation_trace_idx].x = annotation_time
                fig.data[annotation_trace_idx].y = numeric_annotations

        # Update heart rate
        fig.data[hr_trace_idx].x = rate_time_window
        fig.data[hr_trace_idx].y = rate_signal

        # Update mean heart rate
        mean_hr = np.nanmean(rate_signal)
        fig.data[mean_hr_trace_idx].x = [rate_time_window[0], rate_time_window[-1]]
        fig.data[mean_hr_trace_idx].y = [mean_hr, mean_hr]
        fig.data[mean_hr_trace_idx].name = f"Mean HR ({mean_hr:.1f} BPM)"

        # Update IBI plot
        # Only plot non-zero IBI values
        valid_ibi_mask = ibi_signal > 0
        valid_times = time_window[valid_ibi_mask]
        valid_ibis = ibi_signal[valid_ibi_mask]

        fig.data[ibi_trace_idx].x = valid_times
        fig.data[ibi_trace_idx].y = valid_ibis

        # Update threshold lines
        fig.data[min_threshold_trace_idx].x = [
            time_window[0],
            time_window[-1],
        ]  # Min threshold
        fig.data[min_threshold_trace_idx].y = [300, 300]  # 300ms threshold

        fig.data[max_threshold_trace_idx].x = [
            time_window[0],
            time_window[-1],
        ]  # Max threshold
        fig.data[max_threshold_trace_idx].y = [2000, 2000]  # 2000ms threshold


# Create the interactive plot
def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        update_plot(start_slider.value, window_size.value)


# Link the widgets to the update function
start_slider.observe(on_change)
window_size.observe(on_change)

# Display widgets and initial plot
display(widgets.HBox([start_slider, window_size]))
display(fig)

# Initialize the plot
update_plot(start_slider.value, window_size.value)

FigureWidget({
    'data': [{'line': {'color': 'blue'},
              'name': 'ECG Filtered',
              'type': 'scatter',
              'uid': '8f6d248f-a0ee-4a71-8354-98418ec7e293',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'marker': {'color': 'green', 'size': 8, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Peaks',
              'type': 'scatter',
              'uid': '1a8965b8-cf10-4fed-acc6-7ff98d806fcf',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'marker': {'color': 'red', 'size': 8, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Artifactual Peaks',
              'type': 'scatter',
              'uid': 'd36a40d8-0473-4019-983c-efaa1960021a',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'blue', 'shape': 'hv'},
              'mode': 'lines',
              'name': 'Sleep Stage',
              'type': 'scatter',
        

No artifactual peaks found (checked 68 peaks, artifactual peaks signal has 9 ones)
